In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
PJME_data = pd.read_csv(
    "processed_data/PJME_phase2_preprocessed.csv",
    parse_dates=["Datetime"]
)

In [4]:
PJME_data = PJME_data.set_index("Datetime")

# Sort chronologically
PJME_data = PJME_data.sort_index()

print(PJME_data.shape)
print(PJME_data.head())

(145224, 15)
                     PJME_MW  Hour  Day  Week  Month  DayOfWeek  IsWeekend  \
Datetime                                                                     
2002-01-08 01:00:00  29445.0     1    8     2      1          1          0   
2002-01-08 02:00:00  28670.0     2    8     2      1          1          0   
2002-01-08 03:00:00  28375.0     3    8     2      1          1          0   
2002-01-08 04:00:00  28542.0     4    8     2      1          1          0   
2002-01-08 05:00:00  29261.0     5    8     2      1          1          0   

                       Lag_1   Lag_24   Lag_48  Lag_168  Rolling_Mean_24  \
Datetime                                                                   
2002-01-08 01:00:00  31187.0  26862.0  27100.0  30393.0     33452.583333   
2002-01-08 02:00:00  29445.0  25976.0  26097.0  29265.0     33560.208333   
2002-01-08 03:00:00  28670.0  25641.0  25793.0  28357.0     33672.458333   
2002-01-08 04:00:00  28375.0  25666.0  25657.0  27899.0     

In [5]:
# Make sure data is chronological
PJME_data = PJME_data.sort_index()

n = len(PJME_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = PJME_data.iloc[:train_end].copy()
validation = PJME_data.iloc[train_end:val_end].copy()
test = PJME_data.iloc[val_end:].copy()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)


print(train.index.min(), "to", train.index.max())
print(validation.index.min(), "to", validation.index.max())
print(test.index.min(), "to", test.index.max())

Train: (101656, 15)
Validation: (21784, 15)
Test: (21784, 15)
2002-01-08 01:00:00 to 2013-08-13 16:00:00
2013-08-13 17:00:00 to 2016-02-07 08:00:00
2016-02-07 09:00:00 to 2018-08-03 00:00:00


In [6]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "Lag_1",
    "Lag_24",
    "Lag_48",
    "Lag_168",
    "Rolling_Mean_24",
    "Rolling_Mean_168",
    "Rolling_Std_24",
    "Rolling_Std_168"
]

target = "PJME_MW"

X_train = train[features].copy()
X_val = validation[features].copy()
X_test = test[features].copy()

y_train = train[target].copy()
y_val = validation[target].copy()
y_test = test[target].copy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (101656, 15)
X_val: (21784, 15)
X_test: (21784, 15)


In [7]:
from sklearn.preprocessing import StandardScaler

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)
X_val_scaled = feature_scaler.transform(X_val)
X_test_scaled = feature_scaler.transform(X_test)

y_train_scaled = target_scaler.fit_transform(
    y_train.values.reshape(-1, 1)
)

y_val_scaled = target_scaler.transform(
    y_val.values.reshape(-1, 1)
)

y_test_scaled = target_scaler.transform(
    y_test.values.reshape(-1, 1)
)

print("Scaling completed.")

Scaling completed.


In [8]:
def create_sequences(X, y, input_steps=24, output_steps=24):

    X_seq = []
    y_seq = []

    for i in range(input_steps, len(X) - output_steps + 1):

        X_seq.append(
            X[i-input_steps:i]
        )

        y_seq.append(
            y[i:i+output_steps].flatten()
        )

    return np.array(X_seq), np.array(y_seq)


X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    input_steps=24,
    output_steps=24
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    input_steps=24,
    output_steps=24
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    input_steps=24,
    output_steps=24
)


print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val:", X_val_seq.shape)
print("y_val:", y_val_seq.shape)

print("X_test:", X_test_seq.shape)
print("y_test:", y_test_seq.shape)

X_train: (101609, 24, 15)
y_train: (101609, 24)
X_val: (21737, 24, 15)
y_val: (21737, 24)
X_test: (21737, 24, 15)
y_test: (21737, 24)


In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [10]:
bilstm_24_model = Sequential([

    Bidirectional(
        LSTM(
            64,
            return_sequences=False
        ),
        input_shape=(24, X_train_seq.shape[2])
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

bilstm_24_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

bilstm_24_model.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 128)            │        40,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,776 (198.34 KB)

 Trainable params: 50,776 (198.34 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

In [12]:
bilstm_24_history = bilstm_24_model.fit(
    X_train_seq,
    y_train_seq,

    validation_data=(
        X_val_seq,
        y_val_seq
    ),

    epochs=15,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 13ms/step - loss: 0.1070 - mae: 0.2316 - val_loss: 0.0845 - val_mae: 0.2097
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0662 - mae: 0.1841 - val_loss: 0.0809 - val_mae: 0.2035
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0603 - mae: 0.1742 - val_loss: 0.0772 - val_mae: 0.1971
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0561 - mae: 0.1675 - val_loss: 0.0745 - val_mae: 0.1930
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0520 - mae: 0.1614 - val_loss: 0.0845 - val_mae: 0.2005
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0485 - mae: 0.1563 - val_loss: 0.0831 - val_mae: 0.2008
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0452 - mae: 0.1517 - val_loss: 0.0836 - val_mae: 0.2021
Epoch 7: early stopping
Restoring model weights from the end of the best epoch: 4.


In [13]:
bilstm_24_pred_scaled = bilstm_24_model.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:",bilstm_24_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step
Prediction shape: (21737, 24)


In [14]:
bilstm_24_pred = target_scaler.inverse_transform(
    bilstm_24_pred_scaled.reshape(-1, 1)
).reshape(bilstm_24_pred_scaled.shape)

bilstm_24_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

actual_flat = bilstm_24_actual.flatten()
pred_flat = bilstm_24_pred.flatten()

bilstm_24_mae = mean_absolute_error(
    actual_flat,
    pred_flat
)

bilstm_24_rmse = np.sqrt(
    mean_squared_error(
        actual_flat,
        pred_flat
    )
)

non_zero = actual_flat != 0

bilstm_24_mape = np.mean(
    np.abs(
        (actual_flat[non_zero] - pred_flat[non_zero])
        / actual_flat[non_zero]
    )
) * 100

bilstm_24_r2 = r2_score(
    actual_flat,
    pred_flat
)

bilstm_24_bias = np.mean(
    pred_flat - actual_flat
)



print("=== BI-LSTM + earlystopping===")
print("MAE :", bilstm_24_mae)
print("RMSE:", bilstm_24_rmse)
print("MAPE:", bilstm_24_mape)
print("R²  :", bilstm_24_r2)
print("Bias:", bilstm_24_bias)

=== BI-LSTM + earlystopping===
MAE : 1396.5542879780994
RMSE: 1976.8006255034368
MAPE: 4.400636257747516
R²  : 0.9058456082441985
Bias: 98.70227465787741


In [17]:
print("Epochs completed:", len(bilstm_24_history.history["loss"]))

Epochs completed: 7


**Dropout**

In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout


bilstm_drop_model = Sequential([

    Bidirectional(
        LSTM(
            64,
            return_sequences=False
        ),
        input_shape=(24, 15)
    ),
    Dropout(0.2),

    Dense(64, activation="relu"),
    Dropout(0.2),

    Dense(24)
])

bilstm_drop_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

bilstm_drop_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_2 (Bidirectional) │ (None, 128)            │        40,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,776 (198.34 KB)

 Trainable params: 50,776 (198.34 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
bilstm_history_dropout = bilstm_drop_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 13ms/step - loss: 0.1689 - mae: 0.3062 - val_loss: 0.0922 - val_mae: 0.2246
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.1115 - mae: 0.2512 - val_loss: 0.0834 - val_mae: 0.2107
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.1045 - mae: 0.2426 - val_loss: 0.0819 - val_mae: 0.2102
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.1011 - mae: 0.2385 - val_loss: 0.0781 - val_mae: 0.2041
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0979 - mae: 0.2349 - val_loss: 0.0803 - val_mae: 0.2088
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0965 - mae: 0.2330 - val_loss: 0.0797 - val_mae: 0.2050
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0948 - mae: 0.2311 - val_loss: 0.0771 - val_mae: 0.2029
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 15ms/step - loss: 0.0924 - mae: 0.2284 - val_loss: 0.0743 - val_mae: 0.1961
Epoch 9/15
1588/1588 ━━━

In [22]:
dropout_pred_scaled = bilstm_drop_model.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step


In [23]:
dropout_pred = target_scaler.inverse_transform(
    dropout_pred_scaled.reshape(-1, 1)
).reshape(dropout_pred_scaled.shape)

dropout_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [24]:
mae = mean_absolute_error(
    dropout_actual.flatten(),
    dropout_pred.flatten()
)

rmse = np.sqrt(
    mean_squared_error(
        dropout_actual.flatten(),
        dropout_pred.flatten()
    )
)

mape = np.mean(
    np.abs(
        (dropout_actual.flatten() - dropout_pred.flatten())
        / dropout_actual.flatten()
    )
) * 100

r2 = r2_score(
    dropout_actual.flatten(),
    dropout_pred.flatten()
)

bias = np.mean(
    dropout_pred.flatten() -
    dropout_actual.flatten()
)

print("== BILSTM + DROPOUT ==")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

== BILSTM + DROPOUT ==
MAE : 1502.53017196027
RMSE: 2084.7860169122196
MAPE: 4.818527820358442
R²  : 0.8952780275837051
Bias: 442.7513808802807


**Batch Normalization**

In [25]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import  BatchNormalization
bilstm_bn_model = Sequential([

    Bidirectional(
        LSTM(
            64,
            return_sequences=False
        ),
        input_shape=(24, X_train_seq.shape[2])
    ),
    BatchNormalization(),

    Dense(64, activation="relu"),
    BatchNormalization(),

    Dense(24)
])

bilstm_bn_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

bilstm_bn_model.summary()



/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_3 (Bidirectional) │ (None, 128)            │        40,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 51,544 (201.34 KB)

 Trainable params: 51,160 (199.84 KB)

 Non-trainable params: 384 (1.50 KB)

In [26]:
history_bn = bilstm_bn_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 13ms/step - loss: 0.1557 - mae: 0.2893 - val_loss: 0.0941 - val_mae: 0.2257
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0929 - mae: 0.2289 - val_loss: 0.0813 - val_mae: 0.2083
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0849 - mae: 0.2185 - val_loss: 0.0847 - val_mae: 0.2107
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0783 - mae: 0.2095 - val_loss: 0.0814 - val_mae: 0.2017
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0745 - mae: 0.2045 - val_loss: 0.0838 - val_mae: 0.2033
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0695 - mae: 0.1974 - val_loss: 0.0795 - val_mae: 0.2014
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0671 - mae: 0.1949 - val_loss: 0.0835 - val_mae: 0.2004
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0634 - mae: 0.1902 - val_loss: 0.0838 - val_mae: 0.2041
Epoch 9/15
1588/1588 ━━━

In [27]:
bn_pred_scaled = bilstm_bn_model.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step


In [28]:
lstm_bn_pred = target_scaler.inverse_transform(
    bn_pred_scaled.reshape(-1, 1)
).reshape(bn_pred_scaled.shape)

lstm_bn_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [30]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(
    lstm_bn_actual.flatten(),
    lstm_bn_pred.flatten()
)

rmse = np.sqrt(
    mean_squared_error(
        lstm_bn_actual.flatten(),
        lstm_bn_pred.flatten()
    )
)

mape = np.mean(
    np.abs(
        (lstm_bn_actual.flatten() - lstm_bn_pred.flatten())
        / lstm_bn_actual.flatten()
    )
) * 100

r2 = r2_score(
    lstm_bn_actual.flatten(),
    lstm_bn_pred.flatten()
)

bias = np.mean(
    lstm_bn_pred.flatten() - lstm_bn_actual.flatten()
)

print("== BILSTM + Batch ==")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

== BILSTM + Batch ==
MAE : 1511.7297426103162
RMSE: 2172.6960275145207
MAPE: 4.76830623689913
R²  : 0.8862601154385207
Bias: 260.8620456313508


**RMSPROP**

In [31]:
from tensorflow.keras.optimizers import RMSprop

bilstm_rmsprop = Sequential([

    Bidirectional(
        LSTM(
            64,
            return_sequences=False
        ),
        input_shape=(24, X_train_seq.shape[2])
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

bilstm_rmsprop.compile(
    optimizer=RMSprop(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

bilstm_rmsprop.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_4 (Bidirectional) │ (None, 128)            │        40,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,776 (198.34 KB)

 Trainable params: 50,776 (198.34 KB)

 Non-trainable params: 0 (0.00 B)

In [32]:
history_rmsprop = bilstm_rmsprop.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.1058 - mae: 0.2335 - val_loss: 0.0896 - val_mae: 0.2198
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0700 - mae: 0.1896 - val_loss: 0.0841 - val_mae: 0.2094
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0636 - mae: 0.1794 - val_loss: 0.0764 - val_mae: 0.1946
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0591 - mae: 0.1724 - val_loss: 0.0778 - val_mae: 0.1989
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0554 - mae: 0.1669 - val_loss: 0.0829 - val_mae: 0.2079
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0520 - mae: 0.1619 - val_loss: 0.0788 - val_mae: 0.1993
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0490 - mae: 0.1575 - val_loss: 0.0768 - val_mae: 0.1949
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0460 - mae: 0.1531 - val_loss: 0.0806 - val_mae: 0.1992
Epoch 9/15
1588/1588 ━━━

In [33]:
rmsprop_pred_scaled = bilstm_rmsprop.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", rmsprop_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step
Prediction shape: (21737, 24)


In [34]:
rmsprop_pred = target_scaler.inverse_transform(
    rmsprop_pred_scaled.reshape(-1, 1)
).reshape(rmsprop_pred_scaled.shape)

rmsprop_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [36]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rmsprop_mae = mean_absolute_error(
    rmsprop_actual.flatten(),
    rmsprop_pred.flatten()
)

rmsprop_rmse = np.sqrt(
    mean_squared_error(
        rmsprop_actual.flatten(),
        rmsprop_pred.flatten()
    )
)

rmsprop_mape = np.mean(
    np.abs(
        (rmsprop_actual.flatten() - rmsprop_pred.flatten())
        / rmsprop_actual.flatten()
    )
) * 100

rmsprop_r2 = r2_score(
    rmsprop_actual.flatten(),
    rmsprop_pred.flatten()
)

rmsprop_bias = np.mean(
    rmsprop_pred.flatten() -
    rmsprop_actual.flatten()
)

print("=== BILSTM + RMSPROP ===")
print("MAE :", rmsprop_mae)
print("RMSE:", rmsprop_rmse)
print("MAPE:", rmsprop_mape)
print("R²  :", rmsprop_r2)
print("Bias:", rmsprop_bias)

=== BILSTM + RMSPROP ===
MAE : 1512.8497295677685
RMSE: 2182.3433907105477
MAPE: 4.7788321148020145
R²  : 0.8852478006961355
Bias: 414.77281661884115


**SGD**

In [37]:
from tensorflow.keras.optimizers import SGD

bilstm_sgd = Sequential([

    Bidirectional(
        LSTM(
            64,
            return_sequences=False
        ),
        input_shape=(24, X_train_seq.shape[2])
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

bilstm_sgd.compile(
    optimizer=SGD(
        learning_rate=0.001,
        momentum=0.9
    ),
    loss="mse",
    metrics=["mae"]
)

bilstm_sgd.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_5 (Bidirectional) │ (None, 128)            │        40,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,776 (198.34 KB)

 Trainable params: 50,776 (198.34 KB)

 Non-trainable params: 0 (0.00 B)

In [38]:
history_sgd = bilstm_sgd.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 24s 14ms/step - loss: 0.4903 - mae: 0.5413 - val_loss: 0.2470 - val_mae: 0.3953
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.2005 - mae: 0.3541 - val_loss: 0.1814 - val_mae: 0.3338
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.1607 - mae: 0.3150 - val_loss: 0.1592 - val_mae: 0.3099
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.1402 - mae: 0.2919 - val_loss: 0.1451 - val_mae: 0.2927
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1264 - mae: 0.2745 - val_loss: 0.1348 - val_mae: 0.2802
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1167 - mae: 0.2617 - val_loss: 0.1286 - val_mae: 0.2717
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.1098 - mae: 0.2522 - val_loss: 0.1222 - val_mae: 0.2636
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.1048 - mae: 0.2451 - val_loss: 0.1178 - val_mae: 0.2585
Epoch 9/15
1588/1588 ━━━

In [39]:
sgd_pred_scaled = bilstm_sgd.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", sgd_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
Prediction shape: (21737, 24)


In [40]:
sgd_pred = target_scaler.inverse_transform(
    sgd_pred_scaled.reshape(-1, 1)
).reshape(sgd_pred_scaled.shape)

sgd_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [42]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

sgd_mae = mean_absolute_error(
    sgd_actual.flatten(),
    sgd_pred.flatten()
)

sgd_rmse = np.sqrt(
    mean_squared_error(
        sgd_actual.flatten(),
        sgd_pred.flatten()
    )
)

sgd_mape = np.mean(
    np.abs(
        (sgd_actual.flatten() - sgd_pred.flatten())
        / sgd_actual.flatten()
    )
) * 100

sgd_r2 = r2_score(
    sgd_actual.flatten(),
    sgd_pred.flatten()
)

sgd_bias = np.mean(
    sgd_pred.flatten() -
    sgd_actual.flatten()
)
print("=== BILSTM + SGD ===")
print("MAE :", sgd_mae)
print("RMSE:", sgd_rmse)
print("MAPE:", sgd_mape)
print("R²  :", sgd_r2)
print("Bias:", sgd_bias)

=== BILSTM + SGD ===
MAE : 1636.4748680754828
RMSE: 2221.368614750217
MAPE: 5.197385031146546
R²  : 0.8811070493027587
Bias: 53.065937287049444


Learning Rate Scheduling

In [46]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.optimizers import RMSprop

bilstm_lr_model = Sequential([

    Bidirectional(
        LSTM(
            64,
            return_sequences=False
        ),
        input_shape=(24, X_train_seq.shape[2])
    ),

    Dense(64, activation="relu"),
    Dense(24)
])

bilstm_lr_model.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

In [47]:
history_lr = bilstm_lr_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    callbacks=[lr_scheduler],
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.1052 - mae: 0.2334 - val_loss: 0.0919 - val_mae: 0.2250 - learning_rate: 0.0010
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0699 - mae: 0.1899 - val_loss: 0.0885 - val_mae: 0.2161 - learning_rate: 0.0010
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0632 - mae: 0.1790 - val_loss: 0.0770 - val_mae: 0.1993 - learning_rate: 0.0010
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0589 - mae: 0.1723 - val_loss: 0.0797 - val_mae: 0.2042 - learning_rate: 0.0010
Epoch 5/15
1585/1588 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0553 - mae: 0.1668
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 15ms/step - loss: 0.0553 - mae: 0.1668 - val_loss: 0.0848 - val_mae: 0.2110 - learning_rate: 0.0010
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0480 - mae: 0.1548 - val_loss: 0.0767 - val_mae: 0.195

In [48]:
lr_pred_scaled = bilstm_lr_model.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", lr_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step
Prediction shape: (21737, 24)


In [49]:
lr_pred = target_scaler.inverse_transform(
    lr_pred_scaled.reshape(-1, 1)
).reshape(lr_pred_scaled.shape)

lr_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [50]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

lr_mae = mean_absolute_error(
    lr_actual.flatten(),
    lr_pred.flatten()
)

lr_rmse = np.sqrt(
    mean_squared_error(
        lr_actual.flatten(),
        lr_pred.flatten()
    )
)

lr_mape = np.mean(
    np.abs(
        (lr_actual.flatten() - lr_pred.flatten())
        / lr_actual.flatten()
    )
) * 100

lr_r2 = r2_score(
    lr_actual.flatten(),
    lr_pred.flatten()
)

lr_bias = np.mean(
    lr_pred.flatten() -
    lr_actual.flatten()
)

print("==== BILSTM + RMSPROP + LearningRate ====")
print("MAE :", lr_mae)
print("RMSE:", lr_rmse)
print("MAPE:", lr_mape)
print("R²  :", lr_r2)
print("Bias:", lr_bias)

==== BILSTM + RMSPROP + LearningRate ====
MAE : 1420.5269825502144
RMSE: 2015.66074232821
MAPE: 4.485102557417481
R²  : 0.9021074329156991
Bias: 212.6783104496845


**Layers**

In [51]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Bidirectional

bilstm_deep = Sequential([

    Bidirectional(
        LSTM(
            64,
            return_sequences=True
        ),
        input_shape=(48, 15)
    ),

    Bidirectional(
        LSTM(
            32,
            return_sequences=False
        )
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(24)
])

bilstm_deep.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

bilstm_deep.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_9 (Bidirectional) │ (None, 48, 128)        │        40,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_10                │ (None, 64)             │        41,216 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 87,896 (343.34 KB)

 Trainable params: 87,896 (343.34 KB)

 Non-trainable params: 0 (0.00 B)

In [52]:
history_deep = bilstm_deep.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 49s 29ms/step - loss: 0.1034 - mae: 0.2281 - val_loss: 0.0847 - val_mae: 0.2129
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 46s 29ms/step - loss: 0.0649 - mae: 0.1826 - val_loss: 0.0826 - val_mae: 0.2124
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 42s 27ms/step - loss: 0.0578 - mae: 0.1713 - val_loss: 0.0798 - val_mae: 0.2019
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 43s 27ms/step - loss: 0.0519 - mae: 0.1626 - val_loss: 0.0847 - val_mae: 0.2081
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 43s 27ms/step - loss: 0.0460 - mae: 0.1539 - val_loss: 0.0794 - val_mae: 0.1984
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 46s 29ms/step - loss: 0.0403 - mae: 0.1450 - val_loss: 0.0830 - val_mae: 0.2035
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 46s 29ms/step - loss: 0.0356 - mae: 0.1376 - val_loss: 0.0854 - val_mae: 0.2048
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 49s 31ms/step - loss: 0.0311 - mae: 0.1298 - val_loss: 0.0945 - val_mae: 0.2131
Epoch 9/15
1588/1588 ━━━

In [54]:
deep_pred_scaled = bilstm_deep.predict(
    X_test_seq,
    batch_size=64
)
print("Prediction shape:", deep_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step
Prediction shape: (21737, 24)


In [55]:
deep_pred = target_scaler.inverse_transform(
    deep_pred_scaled.reshape(-1, 1)
).reshape(deep_pred_scaled.shape)

deep_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [56]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

deep_mae = mean_absolute_error(
    deep_actual.flatten(),
    deep_pred.flatten()
)

deep_rmse = np.sqrt(
    mean_squared_error(
        deep_actual.flatten(),
        deep_pred.flatten()
    )
)

deep_mape = np.mean(
    np.abs(
        (deep_actual.flatten() - deep_pred.flatten())
        / deep_actual.flatten()
    )
) * 100

deep_r2 = r2_score(
    deep_actual.flatten(),
    deep_pred.flatten()
)

deep_bias = np.mean(
    deep_pred.flatten() -
    deep_actual.flatten()
)

print("==== BILSTM + Layers ==========")
print("MAE :", deep_mae)
print("RMSE:", deep_rmse)
print("MAPE:", deep_mape)
print("R²  :", deep_r2)
print("Bias:", deep_bias)

==== BILSTM + Layers ==========
MAE : 1528.3526456875327
RMSE: 2252.683629134218
MAPE: 4.7733981154444045
R²  : 0.8777313130391772
Bias: 44.36034892682504


**Neurons**

In [57]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Bidirectional

bilstm_128 = Sequential([

    Bidirectional(
        LSTM(
            128,
            input_shape=(48, 15)
        )
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(24)
])

bilstm_128.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

bilstm_128.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_11                │ ?                      │   0 (unbuilt) │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [58]:
history_128 = bilstm_128.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 42s 26ms/step - loss: 0.0954 - mae: 0.2191 - val_loss: 0.0821 - val_mae: 0.2081
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 41s 26ms/step - loss: 0.0635 - mae: 0.1790 - val_loss: 0.0815 - val_mae: 0.2005
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 41s 26ms/step - loss: 0.0571 - mae: 0.1689 - val_loss: 0.0759 - val_mae: 0.1980
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 41s 26ms/step - loss: 0.0513 - mae: 0.1603 - val_loss: 0.0780 - val_mae: 0.1972
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 41s 26ms/step - loss: 0.0456 - mae: 0.1517 - val_loss: 0.0817 - val_mae: 0.2000
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 42s 26ms/step - loss: 0.0395 - mae: 0.1427 - val_loss: 0.0844 - val_mae: 0.2042
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 41s 26ms/step - loss: 0.0339 - mae: 0.1336 - val_loss: 0.0892 - val_mae: 0.2041
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 41s 26ms/step - loss: 0.0290 - mae: 0.1248 - val_loss: 0.0888 - val_mae: 0.2070
Epoch 9/15
1588/1588 ━━━

In [59]:
lstm128_pred_scaled = bilstm_128.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", lstm128_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step
Prediction shape: (21737, 24)


In [60]:
lstm128_pred = target_scaler.inverse_transform(
    lstm128_pred_scaled.reshape(-1, 1)
).reshape(lstm128_pred_scaled.shape)

lstm128_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [62]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

bilstm128_mae = mean_absolute_error(
    lstm128_actual.flatten(),
    lstm128_pred.flatten()
)

bilstm128_rmse = np.sqrt(
    mean_squared_error(
        lstm128_actual.flatten(),
        lstm128_pred.flatten()
    )
)

bilstm128_mape = np.mean(
    np.abs(
        (lstm128_actual.flatten() - lstm128_pred.flatten())
        / lstm128_actual.flatten()
    )
) * 100

bilstm128_r2 = r2_score(
    lstm128_actual.flatten(),
    lstm128_pred.flatten()
)

bilstm128_bias = np.mean(
    lstm128_pred.flatten() -
    lstm128_actual.flatten()
)

print("=== biLSTM 128 NEURONS ===")
print("MAE :", bilstm128_mae)
print("RMSE:", bilstm128_rmse)
print("MAPE:", bilstm128_mape)
print("R²  :", bilstm128_r2)
print("Bias:", bilstm128_bias)

=== biLSTM 128 NEURONS ===
MAE : 1516.9724145784483
RMSE: 2215.393124292808
MAPE: 4.75381377130297
R²  : 0.8817458340046282
Bias: 37.12101231176189


32 **neurons**

In [63]:
bilstm_32 = Sequential([
    
    Bidirectional(
        LSTM(
            32,
            input_shape=(48, 15)
        )
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(24)
])

bilstm_32.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

bilstm_32.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_12                │ ?                      │   0 (unbuilt) │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [64]:
history_32 = bilstm_32.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.1233 - mae: 0.2486 - val_loss: 0.0906 - val_mae: 0.2175
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0700 - mae: 0.1902 - val_loss: 0.0823 - val_mae: 0.2085
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0640 - mae: 0.1802 - val_loss: 0.0839 - val_mae: 0.2077
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0607 - mae: 0.1748 - val_loss: 0.0781 - val_mae: 0.1985
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0580 - mae: 0.1705 - val_loss: 0.0770 - val_mae: 0.1958
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0555 - mae: 0.1667 - val_loss: 0.0798 - val_mae: 0.2024
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0538 - mae: 0.1641 - val_loss: 0.0740 - val_mae: 0.1927
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0521 - mae: 0.1617 - val_loss: 0.0767 - val_mae: 0.1970
Epoch 9/15
1588/1588 ━━━━━━━━━━━

In [65]:
lstm32_pred_scaled = bilstm_32.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", lstm32_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Prediction shape: (21737, 24)


In [66]:
lstm32_pred = target_scaler.inverse_transform(
    lstm32_pred_scaled.reshape(-1, 1)
).reshape(lstm32_pred_scaled.shape)

lstm32_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [67]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

lstm32_mae = mean_absolute_error(
    lstm32_actual.flatten(),
    lstm32_pred.flatten()
)

lstm32_rmse = np.sqrt(
    mean_squared_error(
        lstm32_actual.flatten(),
        lstm32_pred.flatten()
    )
)

lstm32_mape = np.mean(
    np.abs(
        (lstm32_actual.flatten() - lstm32_pred.flatten())
        / lstm32_actual.flatten()
    )
) * 100

lstm32_r2 = r2_score(
    lstm32_actual.flatten(),
    lstm32_pred.flatten()
)

lstm32_bias = np.mean(
    lstm32_pred.flatten() -
    lstm32_actual.flatten()
)

print("=== biLSTM 32 NEURONS ===")
print("MAE :", lstm32_mae)
print("RMSE:", lstm32_rmse)
print("MAPE:", lstm32_mape)
print("R²  :", lstm32_r2)
print("Bias:", lstm32_bias)

=== biLSTM 32 NEURONS ===
MAE : 1451.8358871151795
RMSE: 2088.075172331243
MAPE: 4.594225397717251
R²  : 0.8949473283596974
Bias: 334.6055084348787


Batch Size(32)

In [68]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Bidirectional

bilstm_batch32 = Sequential([
    Bidirectional(
        LSTM(64, input_shape=(48, 15))
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

bilstm_batch32.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history_bilstm_batch32 = bilstm_batch32.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=32,
    verbose=1
)

Epoch 1/15


/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


3176/3176 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0947 - mae: 0.2187 - val_loss: 0.0852 - val_mae: 0.2123
Epoch 2/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - loss: 0.0647 - mae: 0.1806 - val_loss: 0.0764 - val_mae: 0.1972
Epoch 3/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 27s 8ms/step - loss: 0.0587 - mae: 0.1711 - val_loss: 0.0745 - val_mae: 0.1953
Epoch 4/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - loss: 0.0541 - mae: 0.1640 - val_loss: 0.0793 - val_mae: 0.1956
Epoch 5/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 27s 8ms/step - loss: 0.0497 - mae: 0.1575 - val_loss: 0.0782 - val_mae: 0.1965
Epoch 6/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 27s 8ms/step - loss: 0.0454 - mae: 0.1516 - val_loss: 0.0804 - val_mae: 0.1964
Epoch 7/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 27s 8ms/step - loss: 0.0413 - mae: 0.1456 - val_loss: 0.0837 - val_mae: 0.1983
Epoch 8/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - loss: 0.0377 - mae: 0.1401 - val_loss: 0.0882 - val_mae: 0.2044
Epoch 9/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 2

In [69]:
lstm_batch32_pred_scaled = bilstm_batch32.predict(
    X_test_seq,
    batch_size=32
)

print("Prediction shape:", lstm_batch32_pred_scaled.shape)

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
Prediction shape: (21737, 24)


In [70]:
lstmbatch32_pred = target_scaler.inverse_transform(
    lstm_batch32_pred_scaled.reshape(-1, 1)
).reshape(lstm_batch32_pred_scaled.shape)

lstmbatch32_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [71]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

lstmb32_mae = mean_absolute_error(
    lstmbatch32_actual.flatten(),
    lstm32_pred.flatten()
)

lstmb32_rmse = np.sqrt(
    mean_squared_error(
        lstmbatch32_actual.flatten(),
        lstmbatch32_pred.flatten()
    )
)

lstmb32_mape = np.mean(
    np.abs(
        (lstmbatch32_actual.flatten() - lstmbatch32_pred.flatten())
        / lstmbatch32_actual.flatten()
    )
) * 100

lstmb32_r2 = r2_score(
    lstmbatch32_actual.flatten(),
    lstmbatch32_pred.flatten()
)

lstmb32_bias = np.mean(
    lstmbatch32_pred.flatten() -
    lstmbatch32_actual.flatten()
)

print("=== BILSTM batch32 NEURONS ===")
print("MAE :", lstmb32_mae)
print("RMSE:", lstmb32_rmse)
print("MAPE:", lstmb32_mape)
print("R²  :", lstmb32_r2)
print("Bias:", lstmb32_bias)

=== BILSTM batch32 NEURONS ===
MAE : 1451.8358871151795
RMSE: 2212.946221288073
MAPE: 4.694647362033974
R²  : 0.8820069133351793
Bias: 131.2787171987256


BatchSize(128)

In [72]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Bidirectional

bilstm_batch128 = Sequential([
    Bidirectional(
        LSTM(64, input_shape=(48, 15))
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

bilstm_batch128.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history_bilstm_batch128 = bilstm_batch128.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=128,
    verbose=1
)

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 18s 21ms/step - loss: 0.1270 - mae: 0.2525 - val_loss: 0.0907 - val_mae: 0.2179
Epoch 2/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - loss: 0.0696 - mae: 0.1901 - val_loss: 0.0831 - val_mae: 0.2063
Epoch 3/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - loss: 0.0629 - mae: 0.1791 - val_loss: 0.0800 - val_mae: 0.2015
Epoch 4/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - loss: 0.0586 - mae: 0.1720 - val_loss: 0.0813 - val_mae: 0.2058
Epoch 5/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - loss: 0.0556 - mae: 0.1675 - val_loss: 0.0807 - val_mae: 0.2006
Epoch 6/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 16s 21ms/step - loss: 0.0526 - mae: 0.1628 - val_loss: 0.0785 - val_mae: 0.1973
Epoch 7/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - loss: 0.0498 - mae: 0.1589 - val_loss: 0.0798 - val_mae: 0.1995
Epoch 8/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - loss: 0.0473 - mae: 0.1550 - val_loss: 0.0832 - val_mae: 0.2027
Epoch 9/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 

In [73]:
lstm_batch128_pred_scaled = bilstm_batch128.predict(
    X_test_seq,
    batch_size=128
)

print("Prediction shape:", lstm_batch128_pred_scaled.shape)

170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
Prediction shape: (21737, 24)


In [74]:
lstmbatch128_pred = target_scaler.inverse_transform(
    lstm_batch128_pred_scaled.reshape(-1, 1)
).reshape(lstm_batch128_pred_scaled.shape)

lstmbatch128_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [75]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

lstmb128_mae = mean_absolute_error(
    lstmbatch128_actual.flatten(),
    lstmbatch128_pred.flatten()
)

lstmb128_rmse = np.sqrt(
    mean_squared_error(
        lstmbatch128_actual.flatten(),
        lstmbatch128_pred.flatten()
    )
)

lstmb128_mape = np.mean(
    np.abs(
        (lstmbatch128_actual.flatten() - lstmbatch128_pred.flatten())
        / lstmbatch128_actual.flatten()
    )
) * 100

lstmb128_r2 = r2_score(
    lstmbatch128_actual.flatten(),
    lstmbatch128_pred.flatten()
)

lstmb128_bias = np.mean(
    lstmbatch128_pred.flatten() -
    lstmbatch128_actual.flatten()
)

print("=== biLSTM batch128  ===")
print("MAE :", lstmb128_mae)
print("RMSE:", lstmb128_rmse)
print("MAPE:", lstmb128_mape)
print("R²  :", lstmb128_r2)
print("Bias:", lstmb128_bias)

=== biLSTM batch128  ===
MAE : 1444.0930713324165
RMSE: 2112.429112878803
MAPE: 4.540034782981582
R²  : 0.892482506618173
Bias: 195.4266586040471


**HyperParameter Tuning**

In [76]:
!pip install keras-tuner

zsh:1: command not found: pip


In [77]:
import keras_tuner as kt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [78]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Bidirectional
from tensorflow.keras.optimizers import Adam

def build_bilstm(hp):

    model = Sequential()

    model.add(
        Bidirectional(
            LSTM(
                units=hp.Choice(
                    "rnn_units",
                    values=[32, 64, 128]
                )
            ),
            input_shape=(48, 15)
        )
    )

    model.add(
        Dense(
            units=hp.Choice(
                "dense_units",
                values=[32, 64, 128]
            ),
            activation="relu"
        )
    )

    model.add(Dense(24))

    learning_rate = hp.Choice(
        "learning_rate",
        values=[0.0001, 0.0005, 0.001]
    )

    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [79]:
tuner = kt.RandomSearch(
    build_bilstm,
    objective="val_loss",
    max_trials=6,
    executions_per_trial=1,
    directory="hyperparameter_tuning",
    project_name="lstm_48_to_24"
)

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [80]:
tuner.search(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Trial 6 Complete [00h 02m 46s]
val_loss: 0.07651686668395996

Best val_loss So Far: 0.07422251999378204
Total elapsed time: 00h 26m 31s


In [81]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("BEST HYPERPARAMETERS")
print("RNN Units:",best_hp.get("rnn_units"))
print("Dense Units:",best_hp.get("dense_units"))
print("Learning Rate:",best_hp.get("learning_rate"))

BEST HYPERPARAMETERS
RNN Units: 64
Dense Units: 128
Learning Rate: 0.001


In [82]:
best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 128)            │        40,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         3,096 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 60,568 (236.59 KB)

 Trainable params: 60,568 (236.59 KB)

 Non-trainable params: 0 (0.00 B)

In [83]:
history_best = best_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.0540 - mae: 0.1638 - val_loss: 0.0753 - val_mae: 0.1966
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0503 - mae: 0.1585 - val_loss: 0.0796 - val_mae: 0.1965
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0463 - mae: 0.1525 - val_loss: 0.0828 - val_mae: 0.1997
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0423 - mae: 0.1472 - val_loss: 0.0849 - val_mae: 0.2011
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0390 - mae: 0.1420 - val_loss: 0.0844 - val_mae: 0.2002
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0355 - mae: 0.1365 - val_loss: 0.0867 - val_mae: 0.2055
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0329 - mae: 0.1322 - val_loss: 0.0836 - val_mae: 0.1989
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.0302 - mae: 0.1276 - val_loss: 0.0899 - val_mae: 0.2069
Epoch 9/15
1588/1588 ━━━

In [84]:
best_pred_scaled = best_model.predict(
    X_test_seq,
    batch_size=64
)

best_pred = target_scaler.inverse_transform(
    best_pred_scaled.reshape(-1, 1)
).reshape(best_pred_scaled.shape)

best_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step


In [85]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

best_mae = mean_absolute_error(
    best_actual.flatten(),
    best_pred.flatten()
)

best_rmse = np.sqrt(
    mean_squared_error(
        best_actual.flatten(),
        best_pred.flatten()
    )
)

best_mape = np.mean(
    np.abs(
        (best_actual.flatten() - best_pred.flatten())
        / best_actual.flatten()
    )
) * 100

best_r2 = r2_score(
    best_actual.flatten(),
    best_pred.flatten()
)

best_bias = np.mean(
    best_pred.flatten() -
    best_actual.flatten()
)

print("HYPERPARAMETER TUNED BILSTM")
print("MAE :", best_mae)
print("RMSE:", best_rmse)
print("MAPE:", best_mape)
print("R²  :", best_r2)
print("Bias:", best_bias)

HYPERPARAMETER TUNED BILSTM
MAE : 1536.288582541343
RMSE: 2252.794588768414
MAPE: 4.823039051614531
R²  : 0.8777192676523035
Bias: 326.2654558413326


**Additional Logs**

In [86]:
# Additional lag features

PJME_data["Lag_2"] = PJME_data["PJME_MW"].shift(2)
PJME_data["Lag_3"] = PJME_data["PJME_MW"].shift(3)
PJME_data["Lag_6"] = PJME_data["PJME_MW"].shift(6)
PJME_data["Lag_12"] = PJME_data["PJME_MW"].shift(12)
PJME_data["Lag_72"] = PJME_data["PJME_MW"].shift(72)
PJME_data["Lag_336"] = PJME_data["PJME_MW"].shift(336)

In [87]:
PJME_data = PJME_data.dropna().copy()

print("Missing values:")
print(PJME_data.isnull().sum().sum())

print("Shape:")
print(PJME_data.shape)

Missing values:
0
Shape:
(144888, 21)


In [88]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",

    "Lag_1",
    "Lag_2",
    "Lag_3",
    "Lag_6",
    "Lag_12",
    "Lag_24",
    "Lag_48",
    "Lag_72",
    "Lag_168",
    "Lag_336",

    "Rolling_Mean_24",
    "Rolling_Mean_168",
    "Rolling_Std_24",
    "Rolling_Std_168"
]

print("Number of features:", len(features))

Number of features: 21


In [89]:
n = len(PJME_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_data = PJME_data.iloc[:train_end]
val_data = PJME_data.iloc[train_end:val_end]
test_data = PJME_data.iloc[val_end:]

print("Train:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)

Train: (101421, 21)
Validation: (21733, 21)
Test: (21734, 21)


In [90]:
X_train = train_data[features]
y_train = train_data[["PJME_MW"]]

X_val = val_data[features]
y_val = val_data[["PJME_MW"]]

X_test = test_data[features]
y_test = test_data[["PJME_MW"]]

In [91]:
from sklearn.preprocessing import StandardScaler

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)
X_val_scaled = feature_scaler.transform(X_val)
X_test_scaled = feature_scaler.transform(X_test)

y_train_scaled = target_scaler.fit_transform(y_train)
y_val_scaled = target_scaler.transform(y_val)
y_test_scaled = target_scaler.transform(y_test)

print("Scaling completed.")

Scaling completed.


In [92]:
X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    input_steps=48,
    output_steps=24
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    input_steps=48,
    output_steps=24
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    input_steps=48,
    output_steps=24
)

print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val:", X_val_seq.shape)
print("y_val:", y_val_seq.shape)

print("X_test:", X_test_seq.shape)
print("y_test:", y_test_seq.shape)

X_train: (101350, 48, 21)
y_train: (101350, 24)
X_val: (21662, 48, 21)
y_val: (21662, 24)
X_test: (21663, 48, 21)
y_test: (21663, 24)


In [93]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Bidirectional

bilstm_lags = Sequential([
    Bidirectional(
        LSTM(
            64,
            input_shape=(48, 21)
        )
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(24)
])

bilstm_lags.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

bilstm_lags.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [94]:
history_lags = bilstm_lags.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 40s 24ms/step - loss: 0.1010 - mae: 0.2249 - val_loss: 0.0883 - val_mae: 0.2158
Epoch 2/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 39s 25ms/step - loss: 0.0643 - mae: 0.1813 - val_loss: 0.0796 - val_mae: 0.2023
Epoch 3/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 38s 24ms/step - loss: 0.0577 - mae: 0.1710 - val_loss: 0.0872 - val_mae: 0.2082
Epoch 4/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 39s 25ms/step - loss: 0.0517 - mae: 0.1627 - val_loss: 0.0795 - val_mae: 0.1989
Epoch 5/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 38s 24ms/step - loss: 0.0458 - mae: 0.1542 - val_loss: 0.0837 - val_mae: 0.2050
Epoch 6/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 40s 25ms/step - loss: 0.0404 - mae: 0.1461 - val_loss: 0.0866 - val_mae: 0.2054
Epoch 7/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 37s 24ms/step - loss: 0.0357 - mae: 0.1385 - val_loss: 0.0882 - val_mae: 0.2053
Epoch 8/15
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 36s 23ms/step - loss: 0.0318 - mae: 0.1317 - val_loss: 0.0932 - val_mae: 0.2127
Epoch 9/15
1584/1584 ━━━

In [95]:
pred_scaled = bilstm_lags.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", pred_scaled.shape)

339/339 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step
Prediction shape: (21663, 24)


In [96]:
pred = target_scaler.inverse_transform(
    pred_scaled.reshape(-1, 1)
).reshape(pred_scaled.shape)

actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [97]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

mae = mean_absolute_error(
    actual.flatten(),
    pred.flatten()
)

rmse = np.sqrt(
    mean_squared_error(
        actual.flatten(),
        pred.flatten()
    )
)

mape = np.mean(
    np.abs(
        (actual.flatten() - pred.flatten())
        / actual.flatten()
    )
) * 100

r2 = r2_score(
    actual.flatten(),
    pred.flatten()
)

bias = np.mean(
    pred.flatten() - actual.flatten()
)

print("=== BILSTM + ADDITIONAL LAGS ===")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

=== BILSTM + ADDITIONAL LAGS ===
MAE : 1573.5113647575515
RMSE: 2277.8776222061006
MAPE: 4.934510941992853
R²  : 0.875091535405415
Bias: 108.93491731758812


In [98]:

import pandas as pd
import os

bilstm_48_24_results = [
    ["Bi-LSTM Baseline", 48, 24, 1582.6344591814595, 2276.95754794763, 4.940600257325572, 0.8751614747425135, 128.43057508510645],
    ["Bi-LSTM + EarlyStopping", 48, 24, 1396.5542879780994, 1976.8006255034368, 4.400636257747516, 0.9058456082441985, 98.70227465787741],
    ["Bi-LSTM + Dropout", 48, 24, 1502.53017196027, 2084.7860169122196, 4.818527820358442, 0.8952780275837051, 442.7513808802807],
    ["Bi-LSTM + Batch Normalization", 48, 24, 1511.7297426103162, 2172.6960275145207, 4.76830623689913, 0.8862601154385207, 260.8620456313508],
    ["Bi-LSTM + RMSprop", 48, 24, 1512.8497295677685, 2182.3433907105477, 4.7788321148020145, 0.8852478006961355, 414.77281661884115],
    ["Bi-LSTM + SGD", 48, 24, 1636.4748680754828, 2221.368614750217, 5.197385031146546, 0.8811070493027587, 53.065937287049444],
    ["Bi-LSTM + RMSprop + Learning Rate", 48, 24, 1420.5269825502144, 2015.66074232821, 4.485102557417481, 0.9021074329156991, 212.6783104496845],
    ["Bi-LSTM + Additional Layers", 48, 24, 1528.3526456875327, 2252.683629134218, 4.7733981154444045, 0.8777313130391772, 44.36034892682504],
    ["Bi-LSTM 128 Neurons", 48, 24, 1516.9724145784483, 2215.393124292808, 4.75381377130297, 0.8817458340046282, 37.12101231176189],
    ["Bi-LSTM Batch Size 32", 48, 24, 1451.8358871151795, 2212.946221288073, 4.694647362033974, 0.8820069133351793, 131.2787171987256],
    ["Bi-LSTM Batch Size 128", 48, 24, 1444.0930713324165, 2112.429112878803, 4.540034782981582, 0.892482506618173, 195.4266586040471],
    ["Bi-LSTM Hyperparameter Tuned", 48, 24, 1536.288582541343, 2252.794588768414, 4.823039051614531, 0.8777192676523035, 326.2654558413326],
    ["Bi-LSTM + Additional Lags", 48, 24, 1573.5113647575515, 2277.8776222061006, 4.934510941992853, 0.875091535405415, 108.93491731758812]
]

bilstm_48_24_df = pd.DataFrame(
    bilstm_48_24_results,
    columns=[
        "Model",
        "Input Hours",
        "Output Hours",
        "MAE",
        "RMSE",
        "MAPE (%)",
        "R²",
        "Bias"
    ]
)

display(
    bilstm_48_24_df.style.format({
        "MAE": "{:.6f}",
        "RMSE": "{:.6f}",
        "MAPE (%)": "{:.6f}",
        "R²": "{:.6f}",
        "Bias": "{:.6f}"
    })
)

os.makedirs("comparison", exist_ok=True)

bilstm_48_24_df.to_csv(
    "comparison/BiLSTM_48h_to_24h_results.csv",
    index=False,
    float_format="%.6f"
)

print("Saved: comparison/BiLSTM_48h_to_24h_results.csv")

,Model,Input Hours,Output Hours,MAE,RMSE,MAPE (%),R²,Bias
0,Bi-LSTM Baseline,48,24,1582.634459,2276.957548,4.940600,0.875161,128.430575
1,Bi-LSTM + EarlyStopping,48,24,1396.554288,1976.800626,4.400636,0.905846,98.702275
2,Bi-LSTM + Dropout,48,24,1502.530172,2084.786017,4.818528,0.895278,442.751381
3,Bi-LSTM + Batch Normalization,48,24,1511.729743,2172.696028,4.768306,0.886260,260.862046
4,Bi-LSTM + RMSprop,48,24,1512.849730,2182.343391,4.778832,0.885248,414.772817
5,Bi-LSTM + SGD,48,24,1636.474868,2221.368615,5.197385,0.881107,53.065937
6,Bi-LSTM + RMSprop + Learning Rate,48,24,1420.526983,2015.660742,4.485103,0.902107,212.678310
7,Bi-LSTM + Additional Layers,48,24,1528.352646,2252.683629,4.773398,0.877731,44.360349
8,Bi-LSTM 128 Neurons,48,24,1516.972415,2215.393124,4.753814,0.881746,37.121012
9,Bi-LSTM Batch Size 32,48,24,1451.835887,2212.946221,4.694647,0.882007,131.278717


Saved: comparison/BiLSTM_48h_to_24h_results.csv


In [99]:
best_bilstm_48_24 = bilstm_48_24_df.loc[bilstm_48_24_df["Model"] == "Bi-LSTM + EarlyStopping"]
display(
    best_bilstm_48_24.style.format({
        "MAE": "{:.6f}",
        "RMSE": "{:.6f}",
        "MAPE (%)": "{:.6f}",
        "R²": "{:.6f}",
        "Bias": "{:.6f}"
    })
)

,Model,Input Hours,Output Hours,MAE,RMSE,MAPE (%),R²,Bias
1,Bi-LSTM + EarlyStopping,48,24,1396.554288,1976.800626,4.400636,0.905846,98.702275
